In [1]:
import os
import cv2
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Load the three models
model1 = torch.hub.load('C:/Users/sarpa/OneDrive/Desktop/yolov5', 'custom',
                        path='C:/Users/sarpa/PycharmProjects/FYP-Arpanjot_Singh/TestingResources/Models/D1/best.pt',
                        source='local')

# Set the path to the folder containing the images
img_folder = 'C:/Users/sarpa/Downloads/Leb/images/'

# Set the path to the COCO labels file
labels_file = 'C:/Users/sarpa/Downloads/Leb/annotations/instances_default.json'

# Load the COCO labels
with open(labels_file, 'r') as f:
    labels = json.load(f)

# Create a dictionary to map the image filename to its ID
id_dict = {img_info['file_name']: img_info['id'] for img_info in labels['images']}

# Create a dictionary to map the category IDs to their names
id2name = {category['id']: category['name'] for category in labels['categories']}

# Create a list to hold the data for each image
data = []

YOLOv5  2023-2-7 Python-3.9.13 torch-1.13.1+cpu CPU

Fusing layers... 
Model summary: 212 layers, 20865057 parameters, 0 gradients
Adding AutoShape... 


In [6]:
# Loop over all images in the folder
for filename in os.listdir(img_folder):
    if filename.lower().endswith('.png'):
        # Load the image
        img_path = os.path.join(img_folder, filename)
        img = cv2.imread(img_path)

        # Make predictions with model1
        results1 = model1(img)
        dat1 = results1.pandas().xyxy[0]
        result1 = dat1.values
        img = np.squeeze(results1.render())

        # Calculate the distance between the predicted bounding box points and the COCO labels bounding box points
        image_id = id_dict[filename]
        distances = []
        for annotation in labels['annotations']:
            if annotation['image_id'] == image_id:
                label_bbox = np.array(annotation['bbox'])  # [xmin, ymin, width, height]
                label_points = np.array(
                    [label_bbox[0], label_bbox[1], label_bbox[0] + label_bbox[2], label_bbox[1] + label_bbox[3]])
                label_center = np.array(
                    [(label_points[0] + label_points[2]) / 2, (label_points[1] + label_points[3]) / 2])
                pred_points = result1[:, :4].astype(int)
                pred_centers = (pred_points[:, :2] + pred_points[:, 2:]) / 2
                dist = np.sqrt(np.sum((pred_centers - label_center) ** 2, axis=1))
                distances.extend(dist)

                # Get the name of the label from the category ID
                label_id = annotation['category_id']
                label_name = id2name[label_id]

                # Get the average distance for this image
                if distances:
                    avg_distance = np.mean(distances)
                else:
                    avg_distance = np.nan

                # Append the data for this image to the list
                data.append({
                    'image_id': image_id,
                    'label': label_name,
                    'distances': dist,
                    'avg_distance': avg_distance
                })

,image_id,label,distances,avg_distance
0,1,Car door close,"[162.79370534513933, 528.7007570450417, 9.0244...",177.832161
1,1,parking,"[498.65385519416174, 25.63156842645413, 507.70...",281.849852
2,1,parking,"[615.1336967684341, 1138.3804043025336, 636.23...",439.807678
3,1,number plate,"[3.249653827717719, 527.2961361986261, 156.925...",382.713374
4,2,Car door close,"[166.61160163686083, 532.8761261306421, 11.798...",180.643424
...,...,...,...,...
95,24,parking,"[501.2432824736906, 29.243088499678002, 511.29...",383.656665
96,25,Car door close,"[162.8725765744499, 530.4897889686472, 7.17329...",233.511888
97,25,number plate,"[5.632994319187668, 527.8422923800251, 160.602...",232.435493
98,25,parking,"[610.8278613488418, 1133.941187275601, 633.020...",419.155808


In [11]:
# Convert the list of data to a Pandas DataFrame
df = pd.DataFrame(data)

# Print the DataFrame
df['avg_distance'].mean()

312.4218487513015

In [2]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv('output_data.csv')
df

,image_id,label,distances,avg_distance
0,1,Car door close,[ 162.79 528.7 9.0244 10.81],177.832161
1,1,parking,[ 498.65 25.632 507.71 511.47],281.849852
2,1,parking,[ 615.13 1138.4 636.24 633.14],439.807678
3,1,number plate,[ 3.2497 527.3 156.93 158.25],382.713374
4,2,Car door close,[ 166.61 532.88 11.799 11.287],180.643424
...,...,...,...,...
95,24,parking,[ 501.24 29.243 511.29 514.25],383.656665
96,25,Car door close,[ 162.87 530.49 7.1733],233.511888
97,25,number plate,[ 5.633 527.84 160.6],232.435493
98,25,parking,[ 610.83 1133.9 633.02],419.155808
